# Kernel and Non-Kernel Learning Under Real-World Data Imperfections

**Graduate Machine Learning — Spring 2026**  
**Integrated, executable coding appendix for all four datasets**

This notebook is a single end-to-end workflow: raw-data investigation, dataset-specific engineering, quality diagnosis, NumPy/SciPy model implementations, kernel experiments, failure analysis, fairness diagnostics, and computational profiling. Its generated CSV tables and figures are designed to feed the final 20–35 page mini-conference paper.

The notebook does not treat the datasets as interchangeable IID matrices. Airbnb is a mixed geographic marketplace, NYC 311 is a drifting event log with censoring, IBM HR is a small imbalanced tabular sample, and Online Retail is a transactional process that must be aggregated through time.


In [ ]:
from pathlib import Path
import json
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.spatial.distance import cdist

from src.scratch_ml import (
    stratified_split_indices, mutual_information_ranking,
    missingness_associations, outlier_comparison,
    near_constant_report, redundant_numeric_report,
    TabularPreprocessor, LogisticRegressionScratch,
    DecisionTreeScratch, classification_metrics,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid")

SEED = 42
RNG = np.random.default_rng(SEED)
DATA = Path("data")
OUT = Path("processed") / "model_tables"
OUT.mkdir(parents=True, exist_ok=True)
CACHE = Path("cache")
CACHE.mkdir(parents=True, exist_ok=True)
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

paths = {
    "airbnb": DATA / "Merged_Airbnb_Listings.csv",
    "nyc311": DATA / "NYC_311_Service_Requests_from_2023_to_Present.csv",
    "hr": DATA / "IBM_HR_Employee_Attrition.csv",
    "retail": DATA / "Online_Retail.xlsx",
}
for name, path in paths.items():
    assert path.exists(), f"Missing {name}: {path}"
    print(f"{name:8s} {path.stat().st_size / 1e6:9.1f} MB  {path}")


## Dataset 1 — Airbnb Open Data

### Engineering decisions

- **Unit:** one listing at one scrape date.
- **Targets:** advertised nightly `price` and known `host_is_superhost` status. Missing target labels are excluded, never imputed.
- **Feature removal:** constants, identifiers/URLs, long free text, fields above 60% missing unless operationally necessary, price proxies, and post-outcome scrape fields.
- **Missing values:** the export preserves numeric gaps and adds four domain-level indicators instead of a separate flag for every correlated review field. Numerical medians and categorical `Missing` levels are learned later from each training split only.
- **Outliers:** invalid non-positive prices are excluded. Legitimate luxury listings remain; `log1p(price)` is the primary modeling target and a locally winsorized target is retained only for robustness analysis.
- **Split integrity:** `host_group_id` is exported solely to keep all listings from one host on the same side of the train/test split. It is never a predictor.


In [ ]:
air_raw = pd.read_csv(paths["airbnb"], low_memory=False)
print("Shape:", air_raw.shape)
display(air_raw[["city", "price", "host_is_superhost", "room_type"]].head())

air_quality = pd.DataFrame({
    "dtype": air_raw.dtypes.astype(str),
    "missing_rate": air_raw.isna().mean(),
    "n_unique": air_raw.nunique(dropna=True),
}).sort_values(["missing_rate", "n_unique"], ascending=[False, True])
display(air_quality.head(25))
print("Exact duplicate rows:", air_raw.duplicated().sum())
print("Duplicate listing IDs:", air_raw.duplicated("id").sum())
print("\nCities:\n", air_raw["city"].value_counts(dropna=False))
print("\nSuperhost target:\n", air_raw["host_is_superhost"].value_counts(dropna=False))


In [ ]:
def parse_money(s):
    return pd.to_numeric(
        s.astype("string").str.replace(r"[$,]", "", regex=True),
        errors="coerce",
    )

def parse_percent(s):
    return pd.to_numeric(
        s.astype("string").str.replace("%", "", regex=False),
        errors="coerce",
    ) / 100.0

air = air_raw.copy()
air["price"] = parse_money(air["price"])
for c in ["host_response_rate", "host_acceptance_rate"]:
    air[c] = parse_percent(air[c])
for c in ["host_since", "first_review", "last_review", "last_scraped"]:
    air[c] = pd.to_datetime(air[c], errors="coerce")

reference_date = air["last_scraped"].max()
air["host_tenure_days"] = (reference_date - air["host_since"]).dt.days
air["listing_age_days"] = (reference_date - air["first_review"]).dt.days
air["days_since_last_review"] = (reference_date - air["last_review"]).dt.days
air["amenity_count"] = air["amenities"].fillna("[]").str.count(",") + air["amenities"].notna().astype(int)
air["bathrooms_from_text"] = pd.to_numeric(
    air["bathrooms_text"].astype("string").str.extract(r"(\d+(?:\.\d+)?)")[0],
    errors="coerce",
)
air["host_is_superhost_target"] = air["host_is_superhost"].map({"t": 1, "f": 0})

# Stable, pre-outcome features. Reviews are retained because they are known at scoring time,
# but experiments should also report a no-review ablation for causal interpretation.
numeric_features = [
    "latitude", "longitude", "accommodates", "bathrooms", "bathrooms_from_text",
    "bedrooms", "beds", "minimum_nights", "maximum_nights",
    "availability_30", "availability_365",
    "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d",
    "review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness",
    "review_scores_checkin", "review_scores_communication", "review_scores_location",
    "review_scores_value", "reviews_per_month", "host_response_rate",
    "host_acceptance_rate", "host_listings_count", "host_total_listings_count",
    "calculated_host_listings_count", "host_tenure_days", "listing_age_days",
    "days_since_last_review", "amenity_count",
]
categorical_features = [
    "city", "neighbourhood_cleansed", "property_type", "room_type",
    "host_response_time", "host_has_profile_pic", "host_identity_verified",
    "instant_bookable", "has_availability",
]
numeric_features = [c for c in numeric_features if c in air]
categorical_features = [c for c in categorical_features if c in air]

explicit_drop_reasons = {
    "identifiers_urls": [c for c in air.columns if c == "id" or c.endswith("_id") or "url" in c or "picture" in c],
    "long_free_text": [c for c in ["name", "description", "neighborhood_overview", "host_about", "amenities"] if c in air],
    "price_proxies": [c for c in air.columns if c.startswith("price_quote_") or c == "estimated_revenue_l365d"],
    "scrape_or_post_outcome": [c for c in ["scrape_id", "last_scraped", "calendar_last_scraped", "calendar_updated", "source", "license"] if c in air],
    "very_high_missingness": air.columns[air.isna().mean().gt(0.60)].tolist(),
    "constant": air.columns[air.nunique(dropna=False).le(1)].tolist(),
}
display(pd.DataFrame({
    "reason": explicit_drop_reasons.keys(),
    "columns": [", ".join(v) for v in explicit_drop_reasons.values()],
}))


In [ ]:
# Preserve missing values for train-only imputation. Four domain indicators
# replace many perfectly correlated per-column flags.
air_features = air[numeric_features + categorical_features].copy()
air_features.insert(0, "host_group_id", air["host_id"].astype("Int64"))
air_features["review_information_missing"] = air["review_scores_rating"].isna().astype("int8")
air_features["host_response_missing"] = air["host_response_rate"].isna().astype("int8")
air_features["capacity_information_missing"] = air[
    ["bathrooms", "bedrooms", "beds"]
].isna().any(axis=1).astype("int8")
air_features["host_profile_missing"] = air["host_since"].isna().astype("int8")

# Non-positive prices are invalid target records; high positive prices are
# retained because rarity alone does not establish an error.
price_valid = air["price"].gt(0)
price_summary = air.loc[price_valid].groupby(["city", "room_type"])["price"].agg(
    ["count", "median", "mean", "max"]
).sort_values("median")
display(price_summary)

def group_winsorize(frame, value, groups, lo=0.005, hi=0.995, min_group=30):
    out = frame[value].copy()
    global_lo, global_hi = frame[value].quantile([lo, hi])
    for _, idx in frame.groupby(groups, dropna=False).groups.items():
        vals = frame.loc[idx, value]
        bounds = vals.quantile([lo, hi]) if vals.notna().sum() >= min_group else (global_lo, global_hi)
        lower, upper = (bounds.iloc[0], bounds.iloc[1]) if hasattr(bounds, "iloc") else bounds
        out.loc[idx] = vals.clip(lower, upper)
    return out

air_targets = pd.DataFrame(index=air.index)
air_targets["price"] = air["price"]
air_targets["log_price"] = np.log1p(air["price"].where(price_valid))
air_targets["price_winsorized"] = group_winsorize(
    air.loc[price_valid], "price", ["city", "room_type"]
).reindex(air.index)
air_targets["superhost"] = air["host_is_superhost_target"]

air_regression = air_features.loc[price_valid].join(
    air_targets.loc[price_valid, ["price", "log_price", "price_winsorized"]]
)
air_classification = air_features.loc[air_targets["superhost"].notna()].join(
    air_targets.loc[air_targets["superhost"].notna(), "superhost"]
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(air_regression["price"], bins=80, ax=axes[0])
axes[0].set_title("Raw nightly price")
sns.histplot(air_regression["log_price"], bins=80, ax=axes[1])
axes[1].set_title("log1p nightly price")
plt.tight_layout()
plt.show()

print("Regression table:", air_regression.shape)
print("Classification table:", air_classification.shape)
print("Feature gaps deferred to train-only preprocessing:", int(air_features.isna().sum().sum()))
air_regression.to_csv(OUT / "airbnb_regression.csv", index=False)
air_classification.to_csv(OUT / "airbnb_superhost_classification.csv", index=False)


### Airbnb handoff

City and room type change both the center and spread of price, so a universal global treatment is inappropriate. The cleaned tables retain this market structure, preserve missingness evidence, and separate host groups during evaluation. Outlier variants are available for sensitivity checks, but the observed positive target remains the ground truth.

---

## Dataset 2 — NYC 311 Service Requests

The source is too large for repeated full-memory loading. One streaming pass computes exact population marginals and a deterministic hash sample; subsequent notebook runs reuse a source-versioned cache.


In [ ]:
NYC_USECOLS = [
    "Unique Key", "Created Date", "Closed Date", "Agency",
    "Problem (formerly Complaint Type)", "Problem Detail (formerly Descriptor)",
    "Location Type", "Incident Zip", "City", "Status", "Borough",
    "Open Data Channel Type", "Latitude", "Longitude",
    "Resolution Description", "Community Board", "Council District",
]
NYC_SAMPLE_RATE = 0.01   # about 34k rows: sufficient for audits, much cheaper to store
NYC_CHUNK_SIZE = 200_000

def stable_hash_mask(values, rate=NYC_SAMPLE_RATE):
    h = pd.util.hash_pandas_object(
        values.astype("string"), index=False
    ).to_numpy(dtype="uint64")
    return (h % 10_000) < int(rate * 10_000)

def normalized(counter):
    s = pd.Series(counter, dtype=float)
    return s / s.sum()

def tv_distance(p, q):
    idx = p.index.union(q.index)
    return 0.5 * (
        p.reindex(idx, fill_value=0) - q.reindex(idx, fill_value=0)
    ).abs().sum()

source_stat = paths["nyc311"].stat()
cache_key = f"{source_stat.st_size}_{source_stat.st_mtime_ns}_{int(NYC_SAMPLE_RATE * 10_000)}"
sample_cache = CACHE / f"nyc311_hash_sample_{cache_key}.pkl"
counts_cache = CACHE / f"nyc311_population_counts_{cache_key}.json"

if sample_cache.exists() and counts_cache.exists():
    nyc = pd.read_pickle(sample_cache)
    metadata = json.loads(counts_cache.read_text(encoding="utf-8"))
    total_rows = int(metadata["total_rows"])
    full_counts = {
        dimension: Counter(values)
        for dimension, values in metadata["full_counts"].items()
    }
    print(f"Loaded cached 311 sample: {len(nyc):,} of {total_rows:,} rows")
else:
    full_counts = {"complaint": Counter(), "borough": Counter(), "month": Counter()}
    sample_parts, total_rows = [], 0
    for chunk_no, chunk in enumerate(pd.read_csv(
        paths["nyc311"], usecols=NYC_USECOLS,
        chunksize=NYC_CHUNK_SIZE, low_memory=False
    )):
        total_rows += len(chunk)
        created = pd.to_datetime(
            chunk["Created Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce"
        )
        full_counts["complaint"].update(
            chunk["Problem (formerly Complaint Type)"].fillna("Missing")
        )
        full_counts["borough"].update(chunk["Borough"].fillna("Missing"))
        full_counts["month"].update(
            created.dt.to_period("M").astype("string").fillna("Missing")
        )
        keep = stable_hash_mask(chunk["Unique Key"])
        if keep.any():
            sample_parts.append(chunk.loc[keep].copy())
        if chunk_no % 5 == 0:
            sampled_so_far = sum(map(len, sample_parts))
            print(f"chunk {chunk_no + 1:>2}: scanned {total_rows:,}; sampled {sampled_so_far:,}")
    nyc = pd.concat(sample_parts, ignore_index=True)
    nyc.to_pickle(sample_cache)
    counts_cache.write_text(json.dumps({
        "total_rows": total_rows,
        "full_counts": {k: dict(v) for k, v in full_counts.items()},
    }), encoding="utf-8")
    print("Cached the deterministic sample and exact population marginals.")

print(
    f"Full rows: {total_rows:,}; sampled rows: {len(nyc):,}; "
    f"realized rate: {len(nyc)/total_rows:.3%}"
)


In [ ]:
# Distribution audit against exact full-file marginals.
sample_created = pd.to_datetime(nyc["Created Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
sample_counts = {
    "complaint": normalized(Counter(nyc["Problem (formerly Complaint Type)"].fillna("Missing"))),
    "borough": normalized(Counter(nyc["Borough"].fillna("Missing"))),
    "month": normalized(Counter(sample_created.dt.to_period("M").astype("string").fillna("Missing"))),
}
tvd = {}
for dimension in full_counts:
    full_p = normalized(full_counts[dimension])
    tvd[dimension] = tv_distance(full_p, sample_counts[dimension])
display(pd.Series(tvd, name="total_variation_distance").to_frame())

before = len(nyc)
exact_row_dupes = int(nyc.duplicated().sum())
duplicate_keys = int(nyc.duplicated("Unique Key").sum())
nyc["_completeness"] = nyc.notna().sum(axis=1)
nyc = (nyc.sort_values(["Unique Key", "_completeness"])
          .drop_duplicates()
          .drop_duplicates("Unique Key", keep="last")
          .drop(columns="_completeness")
          .reset_index(drop=True))
print({"before": before, "exact_row_dupes": exact_row_dupes,
       "duplicate_keys": duplicate_keys, "after": len(nyc)})


In [ ]:
nyc["created_at"] = pd.to_datetime(nyc["Created Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
nyc["closed_at"] = pd.to_datetime(nyc["Closed Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
nyc["resolution_hours"] = (nyc["closed_at"] - nyc["created_at"]).dt.total_seconds() / 3600
nyc.loc[nyc["resolution_hours"] < 0, "resolution_hours"] = np.nan

# Features known at request creation.
nyc["created_hour"] = nyc["created_at"].dt.hour
nyc["created_dayofweek"] = nyc["created_at"].dt.dayofweek
nyc["created_month"] = nyc["created_at"].dt.month
nyc["created_day"] = nyc["created_at"].dt.day
nyc["is_weekend"] = nyc["created_dayofweek"].isin([5, 6]).astype("int8")
nyc["is_business_hours"] = (
    nyc["created_dayofweek"].lt(5) & nyc["created_hour"].between(8, 17)
).astype("int8")
nyc["hour_sin"] = np.sin(2 * np.pi * nyc["created_hour"] / 24)
nyc["hour_cos"] = np.cos(2 * np.pi * nyc["created_hour"] / 24)
nyc["dow_sin"] = np.sin(2 * np.pi * nyc["created_dayofweek"] / 7)
nyc["dow_cos"] = np.cos(2 * np.pi * nyc["created_dayofweek"] / 7)
nyc["days_from_start"] = (nyc["created_at"] - nyc["created_at"].min()).dt.days

# Spatial/time near-duplicate clusters: retain rows but expose report multiplicity.
nyc["lat_round"] = pd.to_numeric(nyc["Latitude"], errors="coerce").round(4)
nyc["lon_round"] = pd.to_numeric(nyc["Longitude"], errors="coerce").round(4)
nyc["created_10min"] = nyc["created_at"].dt.floor("10min")
near_key = [
    "Problem (formerly Complaint Type)", "Problem Detail (formerly Descriptor)",
    "lat_round", "lon_round", "created_10min",
]
nyc["near_duplicate_report_count"] = nyc.groupby(near_key, dropna=False)["Unique Key"].transform("size")
print(nyc["near_duplicate_report_count"].describe(percentiles=[.5, .9, .99, .999]))

# Collapse rare complaint labels to produce a learnable, auditable classification target.
top_complaints = nyc["Problem (formerly Complaint Type)"].value_counts().head(20).index
nyc["complaint_category"] = nyc["Problem (formerly Complaint Type)"].where(
    nyc["Problem (formerly Complaint Type)"].isin(top_complaints), "OTHER"
).fillna("OTHER")

# Cap only the regression robustness target; retain raw duration.
closed_valid = nyc["resolution_hours"].notna()
duration_cap = nyc.loc[closed_valid, "resolution_hours"].quantile(0.995)
nyc["resolution_hours_capped"] = nyc["resolution_hours"].clip(upper=duration_cap)
nyc["log_resolution_hours"] = np.log1p(nyc["resolution_hours"])

leakage_columns = [
    "Closed Date", "closed_at", "Status", "Resolution Description",
    "resolution_hours", "resolution_hours_capped", "log_resolution_hours",
]
nyc_feature_columns = [
    "Agency", "Location Type", "Incident Zip", "City", "Borough",
    "Open Data Channel Type", "Community Board", "Council District",
    "Latitude", "Longitude", "created_hour", "created_dayofweek", "created_month",
    "created_day", "is_weekend", "is_business_hours", "hour_sin", "hour_cos",
    "dow_sin", "dow_cos", "days_from_start", "near_duplicate_report_count",
]
nyc_feature_columns = [c for c in nyc_feature_columns if c in nyc]

nyc_classification = nyc[nyc_feature_columns + ["complaint_category"]].copy()
nyc_regression = nyc.loc[closed_valid, nyc_feature_columns + [
    "resolution_hours", "resolution_hours_capped", "log_resolution_hours"
]].copy()
print("Classification:", nyc_classification.shape, nyc_classification["complaint_category"].value_counts().head())
print("Regression:", nyc_regression.shape, "duration cap:", round(duration_cap, 2), "hours")

nyc_classification.to_csv(OUT / "nyc311_complaint_classification_sample.csv", index=False)
nyc_regression.to_csv(OUT / "nyc311_resolution_regression_sample.csv", index=False)
pd.Series(tvd, name="TVD").to_csv(OUT / "nyc311_sampling_audit.csv")


### NYC 311 handoff

The deterministic 1% sample remains distribution-audited against exact full-file complaint, borough, and month marginals. Open requests are treated as right-censored rather than zero-duration cases. Post-resolution fields are excluded from predictors, and cyclic encodings preserve temporal geometry.

---

## Dataset 3 — IBM HR Employee Attrition

This small, imbalanced dataset needs conservative feature selection and an untouched test set. Constant columns and the employee identifier are removed; driver analysis reports association rather than causality. Resampling occurs after splitting and is retained only as an experimental alternative to the original prevalence.


In [ ]:
hr_raw = pd.read_csv(paths["hr"])
print("Shape:", hr_raw.shape)
print("Missing cells:", hr_raw.isna().sum().sum(), "Exact duplicates:", hr_raw.duplicated().sum())
print(hr_raw["Attrition"].value_counts().to_frame().assign(rate=lambda x: x["count"]/len(hr_raw)))

constant_cols = hr_raw.columns[hr_raw.nunique(dropna=False).le(1)].tolist()
identifier_cols = ["EmployeeNumber"]
drop_hr = constant_cols + identifier_cols
print("Dropped constants:", constant_cols)
print("Dropped identifiers:", identifier_cols)

hr = hr_raw.drop(columns=drop_hr).copy()
hr["Attrition"] = hr["Attrition"].map({"Yes": 1, "No": 0}).astype("int8")

# Domain features expose interpretable ratios while guarding division by zero.
hr["income_per_job_level"] = hr["MonthlyIncome"] / hr["JobLevel"].replace(0, np.nan)
hr["company_tenure_ratio"] = hr["YearsAtCompany"] / hr["TotalWorkingYears"].replace(0, np.nan)
hr["role_tenure_ratio"] = hr["YearsInCurrentRole"] / hr["YearsAtCompany"].replace(0, np.nan)
hr["manager_tenure_ratio"] = hr["YearsWithCurrManager"] / hr["YearsAtCompany"].replace(0, np.nan)
hr["promotion_stagnation_ratio"] = hr["YearsSinceLastPromotion"] / hr["YearsAtCompany"].replace(0, np.nan)
hr = hr.replace([np.inf, -np.inf], np.nan).fillna(0)

X_hr = hr.drop(columns="Attrition")
y_hr = hr["Attrition"]
hr_train_idx, hr_test_idx = stratified_split_indices(y_hr, test_size=0.25, random_state=SEED)
X_hr_train, X_hr_test = X_hr.iloc[hr_train_idx].copy(), X_hr.iloc[hr_test_idx].copy()
y_hr_train, y_hr_test = y_hr.iloc[hr_train_idx].copy(), y_hr.iloc[hr_test_idx].copy()
print("Train/test:", X_hr_train.shape, X_hr_test.shape)


In [ ]:
# Root-driver association audit.
numeric_hr = X_hr.select_dtypes(include=np.number).columns
categorical_hr = X_hr.select_dtypes(exclude=np.number).columns

numeric_effects = {}
for c in numeric_hr:
    yes, no = X_hr.loc[y_hr.eq(1), c], X_hr.loc[y_hr.eq(0), c]
    pooled = np.sqrt((yes.var(ddof=1) + no.var(ddof=1)) / 2)
    numeric_effects[c] = (yes.mean() - no.mean()) / pooled if pooled > 0 else 0
numeric_effects = pd.Series(numeric_effects, name="standardized_mean_difference")

categorical_effects = {}
categorical_details = []
for c in categorical_hr:
    tab = pd.DataFrame({c: X_hr[c], "Attrition": y_hr}).groupby(c)["Attrition"].agg(["mean", "count"])
    categorical_effects[c] = tab["mean"].max() - tab["mean"].min()
    tmp = tab.reset_index().rename(columns={"mean": "attrition_rate"})
    tmp.insert(0, "feature", c)
    tmp = tmp.rename(columns={c: "level"})
    categorical_details.append(tmp)
categorical_effects = pd.Series(categorical_effects, name="attrition_rate_range")

display(numeric_effects.reindex(numeric_effects.abs().sort_values(ascending=False).index).head(15).to_frame())
display(categorical_effects.sort_values(ascending=False).head(15).to_frame())
display(pd.concat(categorical_details).sort_values("attrition_rate", ascending=False).head(20))


In [ ]:
# Training-only mutual-information ranking implemented with empirical
# discretisation. It is a filter ranking, not causal evidence.
mi_ranking = mutual_information_ranking(X_hr_train, y_hr_train)
display(mi_ranking.head(25).to_frame())

# Remove only demonstrably redundant numeric columns at |rho| > .95.
num_train = X_hr_train.select_dtypes(include=np.number)
corr = num_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
redundant_pairs = [(row, col, upper.loc[row, col])
                   for col in upper.columns for row in upper.index
                   if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > .95]
display(pd.DataFrame(redundant_pairs, columns=["feature_1", "feature_2", "abs_correlation"]))


In [ ]:
def random_undersample(X, y, random_state=SEED):
    rng = np.random.default_rng(random_state)
    pos = np.flatnonzero(y.to_numpy() == 1)
    neg = np.flatnonzero(y.to_numpy() == 0)
    keep_neg = rng.choice(neg, size=len(pos), replace=False)
    keep = rng.permutation(np.r_[pos, keep_neg])
    return X.iloc[keep].reset_index(drop=True), y.iloc[keep].reset_index(drop=True)

def mixed_smote(X, y, target_ratio=1.0, k_neighbors=5, random_state=SEED):
    # Mixed-type SMOTE written with NumPy/SciPy only. Numeric values are
    # interpolated; categorical values are inherited from either parent.
    rng = np.random.default_rng(random_state)
    X = X.reset_index(drop=True).copy()
    y = y.reset_index(drop=True).copy()
    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X.columns.difference(num_cols).tolist()
    minority = X.loc[y.eq(1)].reset_index(drop=True)
    n_majority, n_minority = int(y.eq(0).sum()), int(y.eq(1).sum())
    n_new = max(0, int(target_ratio * n_majority) - n_minority)
    if n_new == 0:
        return X, y
    med = minority[num_cols].median()
    scale = (minority[num_cols].quantile(.75) - minority[num_cols].quantile(.25)).replace(0, 1)
    Z = ((minority[num_cols] - med) / scale).to_numpy(float)
    distances = cdist(Z, Z)
    np.fill_diagonal(distances, np.inf)
    neighbors = np.argpartition(distances, min(k_neighbors, len(minority) - 1), axis=1)[:, :k_neighbors]
    synthetic = []
    for _ in range(n_new):
        i = rng.integers(len(minority))
        j = int(rng.choice(neighbors[i]))
        gap = rng.random()
        row = {}
        for c in num_cols:
            row[c] = minority.at[i, c] + gap * (minority.at[j, c] - minority.at[i, c])
        for c in cat_cols:
            row[c] = minority.at[i if rng.random() < .5 else j, c]
        synthetic.append(row)
    X_syn = pd.DataFrame(synthetic, columns=X.columns)
    X_out = pd.concat([X, X_syn], ignore_index=True)
    y_out = pd.concat([y, pd.Series(np.ones(n_new, dtype="int8"), name=y.name)], ignore_index=True)
    return X_out, y_out

X_hr_under, y_hr_under = random_undersample(X_hr_train, y_hr_train)
X_hr_smote, y_hr_smote = mixed_smote(X_hr_train, y_hr_train)

balance_audit = pd.DataFrame({
    "original_train": y_hr_train.value_counts(),
    "undersampled_train": y_hr_under.value_counts(),
    "mixed_smote_train": y_hr_smote.value_counts(),
}).fillna(0).astype(int)
display(balance_audit)

X_hr_train.assign(Attrition=y_hr_train.to_numpy()).to_csv(OUT / "hr_train_original.csv", index=False)
X_hr_under.assign(Attrition=y_hr_under).to_csv(OUT / "hr_train_undersampled.csv", index=False)
X_hr_smote.assign(Attrition=y_hr_smote).to_csv(OUT / "hr_train_mixed_smote.csv", index=False)
X_hr_test.assign(Attrition=y_hr_test.to_numpy()).to_csv(OUT / "hr_test_untouched.csv", index=False)
mi_ranking.to_csv(OUT / "hr_training_only_mutual_information.csv")


### HR engineering conclusion

The strongest associations should be interpreted as candidate drivers to test, not causal effects. The test set is untouched and preserves the real 16% attrition prevalence. Undersampling discards majority information; mixed SMOTE adds assumptions by interpolating minority employees. Later modeling compares both against the original imbalanced training set using macro-F1 and balanced accuracy, which expose minority-class failure hidden by ordinary accuracy.

---

## Dataset 4 — Online Retail Transactions

### From transaction lines to customer behavior

Raw lines are not independent customer examples. We use a chronological observation window and future window:

- **Observation:** transactions before 1 September 2011 → customer features.
- **Future:** 1 September–9 December 2011 → future net spending and future segment.

This prevents using future behavior to predict itself. Missing customer IDs cannot be assigned safely and are excluded from customer modeling (but their volume is audited). Cancellations/returns are separated from purchases rather than silently converted into sales. Exact duplicate source rows are removed, while repeated products within an invoice remain valid quantities.

The future segment is operational:

- `Future_Inactive`: no positive future net spend,
- `Future_Low`, `Future_Mid`, `Future_High`: tertiles among active future customers.

Thus classification asks whether historical behavior predicts a future customer state, while regression predicts continuous future net spending.


In [ ]:
retail_raw = pd.read_excel(paths["retail"], sheet_name="Online Retail")
print("Shape:", retail_raw.shape)
print("Date range:", retail_raw["InvoiceDate"].min(), "to", retail_raw["InvoiceDate"].max())
print("Missing rates:\n", retail_raw.isna().mean().sort_values(ascending=False))
print("Exact duplicates:", retail_raw.duplicated().sum())
print("Cancellation invoice rows:", retail_raw["InvoiceNo"].astype(str).str.startswith("C").sum())
print("Nonpositive quantity:", retail_raw["Quantity"].le(0).sum())
print("Nonpositive unit price:", retail_raw["UnitPrice"].le(0).sum())


In [ ]:
retail = retail_raw.drop_duplicates().copy()
retail["CustomerID"] = pd.to_numeric(retail["CustomerID"], errors="coerce")
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"], errors="coerce")
retail["is_cancel_invoice"] = retail["InvoiceNo"].astype(str).str.startswith("C")
retail["line_value"] = retail["Quantity"] * retail["UnitPrice"]

retail_audit = pd.Series({
    "raw_rows": len(retail_raw),
    "exact_duplicates_removed": retail_raw.duplicated().sum(),
    "missing_customer_rows_removed": retail["CustomerID"].isna().sum(),
    "nonpositive_price_rows_removed": retail["UnitPrice"].le(0).sum(),
    "invalid_date_rows_removed": retail["InvoiceDate"].isna().sum(),
}, name="count")
display(retail_audit.to_frame())

# Price <= 0 cannot define spending; customer/date are required for longitudinal aggregation.
retail = retail.loc[
    retail["CustomerID"].notna()
    & retail["InvoiceDate"].notna()
    & retail["UnitPrice"].gt(0)
].copy()
retail["CustomerID"] = retail["CustomerID"].astype("int64")
retail["is_return"] = retail["Quantity"].lt(0) | retail["is_cancel_invoice"]
retail["purchase_value"] = retail["line_value"].where(~retail["is_return"] & retail["Quantity"].gt(0), 0.0)
retail["return_value"] = (-retail["line_value"]).where(retail["is_return"], 0.0).clip(lower=0)

CUTOFF = pd.Timestamp("2011-09-01")
END = retail["InvoiceDate"].max() + pd.Timedelta(seconds=1)
history = retail.loc[retail["InvoiceDate"].lt(CUTOFF)].copy()
future = retail.loc[retail["InvoiceDate"].ge(CUTOFF) & retail["InvoiceDate"].lt(END)].copy()
print("History:", history.shape, history["InvoiceDate"].min(), "to", history["InvoiceDate"].max())
print("Future:", future.shape, future["InvoiceDate"].min(), "to", future["InvoiceDate"].max())


In [ ]:
def customer_features(tx, cutoff):
    purchases = tx.loc[tx["purchase_value"].gt(0)].copy()
    returns = tx.loc[tx["return_value"].gt(0)].copy()

    invoice = purchases.groupby(["CustomerID", "InvoiceNo"], as_index=False).agg(
        invoice_date=("InvoiceDate", "max"),
        order_value=("purchase_value", "sum"),
        units=("Quantity", "sum"),
        unique_products=("StockCode", "nunique"),
    )
    base = purchases.groupby("CustomerID").agg(
        recency_days=("InvoiceDate", lambda s: (cutoff - s.max()).days),
        tenure_days=("InvoiceDate", lambda s: (s.max() - s.min()).days),
        product_diversity=("StockCode", "nunique"),
        total_units=("Quantity", "sum"),
        gross_spend=("purchase_value", "sum"),
        avg_unit_price=("UnitPrice", "mean"),
        country=("Country", lambda s: s.mode().iat[0] if not s.mode().empty else "Missing"),
        weekend_purchase_share=("InvoiceDate", lambda s: s.dt.dayofweek.isin([5, 6]).mean()),
        evening_purchase_share=("InvoiceDate", lambda s: s.dt.hour.ge(18).mean()),
    )
    order = invoice.groupby("CustomerID").agg(
        frequency=("InvoiceNo", "nunique"),
        avg_order_value=("order_value", "mean"),
        max_order_value=("order_value", "max"),
        avg_basket_units=("units", "mean"),
        avg_products_per_order=("unique_products", "mean"),
    )
    ret = returns.groupby("CustomerID").agg(
        return_count=("InvoiceNo", "size"),
        return_value=("return_value", "sum"),
    )
    out = base.join(order).join(ret, how="left")
    out[["return_count", "return_value"]] = out[["return_count", "return_value"]].fillna(0)
    out["return_value_rate"] = out["return_value"] / (out["gross_spend"] + out["return_value"]).replace(0, np.nan)

    # Spending trend: second half versus first half of the observation window.
    midpoint = tx["InvoiceDate"].min() + (cutoff - tx["InvoiceDate"].min()) / 2
    first = purchases.loc[purchases["InvoiceDate"].lt(midpoint)].groupby("CustomerID")["purchase_value"].sum()
    second = purchases.loc[purchases["InvoiceDate"].ge(midpoint)].groupby("CustomerID")["purchase_value"].sum()
    out["spend_first_half"] = first
    out["spend_second_half"] = second
    out[["spend_first_half", "spend_second_half"]] = out[["spend_first_half", "spend_second_half"]].fillna(0)
    out["spend_trend_ratio"] = (
        (out["spend_second_half"] + 1) / (out["spend_first_half"] + 1)
    )
    # Remove mechanically redundant summaries before distance/kernel models.
    out = out.drop(columns=["return_value"])
    return out.replace([np.inf, -np.inf], np.nan).fillna(0)

retail_features = customer_features(history, CUTOFF)

future_customer = future.groupby("CustomerID").agg(
    future_gross_spend=("purchase_value", "sum"),
    future_return_value=("return_value", "sum"),
    future_orders=("InvoiceNo", lambda s: s[future.loc[s.index, "purchase_value"].gt(0)].nunique()),
)
future_customer["future_net_spending"] = (
    future_customer["future_gross_spend"] - future_customer["future_return_value"]
).clip(lower=0)

customer = retail_features.join(future_customer, how="left")
customer[["future_gross_spend", "future_return_value", "future_orders", "future_net_spending"]] = (
    customer[["future_gross_spend", "future_return_value", "future_orders", "future_net_spending"]].fillna(0)
)

active = customer["future_net_spending"].gt(0)
customer["customer_segment"] = "Future_Inactive"
if active.sum() >= 3:
    # Rank before qcut makes ties deterministic and yields nonempty balanced active tiers.
    customer.loc[active, "customer_segment"] = pd.qcut(
        customer.loc[active, "future_net_spending"].rank(method="first"),
        q=3,
        labels=["Future_Low", "Future_Mid", "Future_High"],
    ).astype("string")

target_cols = [
    "future_gross_spend", "future_return_value", "future_orders",
    "future_net_spending", "customer_segment",
]
print("Customer-level shape:", customer.shape)
display(customer["customer_segment"].value_counts().to_frame())
display(customer["future_net_spending"].describe(percentiles=[.5, .75, .9, .95, .99]).to_frame())


In [ ]:
# Final task tables use only historical features plus the requested target.
historical_cols = retail_features.columns.tolist()
retail_classification = customer[historical_cols + ["customer_segment"]].reset_index()
retail_regression = customer[historical_cols + ["future_net_spending"]].reset_index()
retail_regression["log_future_spending"] = np.log1p(retail_regression["future_net_spending"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(retail_regression["future_net_spending"], bins=60, ax=axes[0])
axes[0].set_title("Future net spending")
sns.histplot(retail_regression["log_future_spending"], bins=60, ax=axes[1])
axes[1].set_title("log1p future net spending")
plt.tight_layout()
plt.show()

retail_classification.to_csv(OUT / "retail_future_segment_classification.csv", index=False)
retail_regression.to_csv(OUT / "retail_future_spending_regression.csv", index=False)
retail_audit.to_csv(OUT / "retail_cleaning_audit.csv")
print("Classification:", retail_classification.shape)
print("Regression:", retail_regression.shape)


### Retail handoff

The raw transaction lines are converted to chronological customer histories. Recency, frequency, monetary, basket, return-rate, timing, and trend features are retained; mechanically redundant summaries are removed before distance and kernel modeling. Future targets are defined strictly after September 1, 2011.

---

## Engineering deliverables

The resulting tables preserve task-specific structure while deferring generic imputation, encoding, and scaling to training folds. This keeps the data-quality investigation connected to the values actually observed and prevents preprocessing leakage.


In [ ]:
outputs = sorted(OUT.glob("*.csv"))
summary = []
for path in outputs:
    size_mb = path.stat().st_size / 1e6
    # Counting lines avoids loading generated files again.
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        rows = sum(1 for _ in f) - 1
    summary.append({"file": path.name, "rows": rows, "size_MB": round(size_mb, 2)})
output_summary = pd.DataFrame(summary)
display(output_summary)
assert len(outputs) >= 10
assert air_regression["price"].gt(0).all()
assert air_classification["superhost"].isin([0, 1]).all()
assert nyc_regression["resolution_hours"].ge(0).all()
assert not set(["Closed Date", "Status", "Resolution Description"]).intersection(nyc_feature_columns)
assert set(y_hr_test.unique()).issubset({0, 1})
assert retail_features.index.is_unique
assert retail_regression["future_net_spending"].ge(0).all()
assert air_regression["host_group_id"].notna().all()
print("All engineering integrity checks passed.")


## Modeling handoff

- **Airbnb:** split by host group; compare raw/log target sensitivity and interpret review features as predictive rather than causal.
- **NYC 311:** use chronological validation; report macro-F1 for complaints and MAE/RMSE for resolution.
- **HR:** preserve the untouched real-prevalence test set; compare original, undersampled, and mixed-SMOTE training variants with macro-F1 and balanced accuracy.
- **Retail:** retain the historical/future cutoff; evaluate segment macro-F1 and future-spending MAE/RMSE while recognizing the zero-spending point mass.

Every later algorithm receives the same task-specific split and a preprocessor fitted only on its training data.


# Rigorous data-quality investigation

## Missingness: what can and cannot be established

MCAR is an independence claim and can be challenged with observed data. For each important missing field, the following tests compare its missingness indicator against observed variables: Mann–Whitney U with rank-biserial effect for numeric variables, and chi-square with Cramér’s V for categorical variables. Significant dependence is evidence **against MCAR**.

MAR versus MNAR cannot be proved from the observed table alone: MNAR depends on the unobserved value itself. Therefore the notebook reports the strongest defensible mechanism—supported MAR, structural missingness, outcome censoring, or suspected MNAR—and states the assumption used for imputation.


In [ ]:
missingness_results = []

air_missing_candidates = ["city", "room_type", "source", "number_of_reviews",
                          "host_listings_count", "availability_365"]
for col in ["price", "host_response_rate", "review_scores_rating",
            "bedrooms", "host_is_superhost"]:
    result = missingness_associations(air, col, air_missing_candidates)
    result.insert(0, "dataset", "Airbnb")
    missingness_results.append(result)

nyc_missing_candidates = ["Agency", "Problem (formerly Complaint Type)", "Borough",
                          "Open Data Channel Type", "Status", "created_hour",
                          "created_dayofweek"]
for col in ["Closed Date", "Latitude", "Incident Zip", "Resolution Description"]:
    result = missingness_associations(nyc, col, nyc_missing_candidates)
    result.insert(0, "dataset", "NYC 311")
    missingness_results.append(result)

retail_quality = retail_raw.copy()
retail_quality["is_cancellation"] = retail_quality["InvoiceNo"].astype(str).str.startswith("C")
retail_quality["invoice_month"] = pd.to_datetime(retail_quality["InvoiceDate"]).dt.month
retail_missing_candidates = ["Country", "Quantity", "UnitPrice", "is_cancellation", "invoice_month"]
for col in ["CustomerID", "Description"]:
    result = missingness_associations(retail_quality, col, retail_missing_candidates)
    result.insert(0, "dataset", "Online Retail")
    missingness_results.append(result)

missingness_tests = pd.concat(missingness_results, ignore_index=True)
display(missingness_tests.groupby(["dataset", "missing_column"]).head(4))

mechanism_assessment = pd.DataFrame([
    ["Airbnb", "host/review fields", "MAR + structural",
     "Missingness varies by city, room type, host activity, and review count.",
     "Preserve gaps and domain indicators; fit median/category handling on each training split."],
    ["Airbnb", "price", "MAR / source-dependent",
     "Availability of advertised price differs with scrape/source context.",
     "Rows without a target are excluded; target labels are never imputed."],
    ["NYC 311", "Closed Date", "MNAR-like right censoring",
     "Missingness is mechanically tied to unresolved status and request age.",
     "Open cases are censored and excluded from ordinary duration regression."],
    ["NYC 311", "geolocation", "MAR",
     "Missingness depends on agency, channel, borough, and address availability.",
     "Categorical location remains; coordinates require indicators/imputation in model folds."],
    ["IBM HR", "all fields", "No observed missingness",
     "Zero missing cells does not prove absence of collection or survivorship bias.",
     "No imputation; audit constants and suspiciously clean schema."],
    ["Online Retail", "CustomerID", "MAR, possibly structural",
     "Anonymous/guest transactions differ by country and invoice behavior.",
     "Exclude from customer-level modeling because identities cannot be reconstructed safely."],
    ["Online Retail", "Description", "MAR",
     "Missingness is associated with stock/price and transaction process.",
     "Description is not needed after behavioral aggregation."],
], columns=["dataset", "field", "assessment", "evidence", "action"])
display(mechanism_assessment)
missingness_tests.to_csv(ARTIFACTS / "missingness_association_tests.csv", index=False)
mechanism_assessment.to_csv(ARTIFACTS / "missingness_mechanism_assessment.csv", index=False)


## Outliers: Z-score versus IQR versus Isolation Forest

The methods answer different questions:

- Z-score assumes a meaningful mean and standard deviation; it is fragile on skewed price, duration, and spending distributions.
- IQR is robust but univariate. It often marks valid luxury listings or high-value customers because rarity is not the same as error.
- Isolation Forest is multivariate and detects unusual feature combinations, but its contamination level is a modeling choice rather than ground truth.

The comparison below uses the same 1% multivariate contamination budget and reports flag counts plus Jaccard overlap. Low overlap is evidence that “outlier” is definition-dependent.


In [ ]:
quality_samples = {}
quality_samples["Airbnb"] = air.loc[air["price"].gt(0), [
    "price", "accommodates", "minimum_nights", "number_of_reviews", "availability_365"
]].dropna().sample(min(5000, air["price"].gt(0).sum()), random_state=SEED)
quality_samples["NYC 311"] = nyc.loc[nyc["resolution_hours"].notna(), [
    "resolution_hours", "Latitude", "Longitude", "near_duplicate_report_count"
]].apply(pd.to_numeric, errors="coerce").dropna().sample(
    min(5000, nyc["resolution_hours"].notna().sum()), random_state=SEED
)
quality_samples["IBM HR"] = hr[[
    "Age", "MonthlyIncome", "DistanceFromHome", "TotalWorkingYears", "YearsAtCompany"
]].sample(min(1470, len(hr)), random_state=SEED)
retail_outlier = retail.copy()
retail_outlier["absolute_line_value"] = retail_outlier["line_value"].abs()
quality_samples["Online Retail"] = retail_outlier[[
    "Quantity", "UnitPrice", "absolute_line_value"
]].replace([np.inf, -np.inf], np.nan).dropna().sample(5000, random_state=SEED)

outlier_tables, anomaly_flags = [], {}
for dataset, sample in quality_samples.items():
    table, flags = outlier_comparison(sample, sample.columns.tolist(),
                                      contamination=.01, random_state=SEED)
    table.insert(0, "dataset", dataset)
    outlier_tables.append(table)
    anomaly_flags[dataset] = flags
outlier_comparison_table = pd.concat(outlier_tables, ignore_index=True)
display(outlier_comparison_table)
outlier_comparison_table.to_csv(ARTIFACTS / "outlier_method_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 5))
plot_data = outlier_comparison_table.melt(
    id_vars=["dataset", "feature"],
    value_vars=["zscore_count", "iqr_count", "isolation_count"],
    var_name="method", value_name="flag_count"
)
sns.barplot(data=plot_data, x="dataset", y="flag_count", hue="method", ax=ax)
ax.set_title("Outlier flags disagree because the distributions are skewed and multivariate")
plt.tight_layout()
plt.savefig(ARTIFACTS / "outlier_method_counts.png", dpi=160)
plt.show()


## Feature and label quality

Feature quality is checked before model fitting. Constant/near-constant variables contribute no stable signal, highly correlated variables create conditioning problems, and post-outcome fields create leakage. Label quality is evaluated separately because a perfectly cleaned feature matrix cannot repair ambiguous or biased ground truth.


In [ ]:
feature_quality_tables = []
for dataset, frame in {
    "Airbnb": air_raw,
    "NYC 311": nyc,
    "IBM HR": hr_raw,
    "Online Retail": retail_raw,
}.items():
    report = near_constant_report(frame)
    report.insert(0, "dataset", dataset)
    feature_quality_tables.append(report)
near_constant_table = pd.concat(feature_quality_tables, ignore_index=True)
display(near_constant_table.loc[near_constant_table["near_constant"]].head(30))

redundancy_tables = []
for dataset, frame in {
    "Airbnb": air_features.drop(columns="host_group_id").sample(min(5000, len(air_features)), random_state=SEED),
    "NYC 311": nyc[nyc_feature_columns].sample(min(5000, len(nyc)), random_state=SEED),
    "IBM HR": X_hr,
    "Online Retail": retail_features,
}.items():
    report = redundant_numeric_report(frame, threshold=.90)
    report.insert(0, "dataset", dataset)
    redundancy_tables.append(report)
redundancy_table = pd.concat(redundancy_tables, ignore_index=True)
display(redundancy_table.head(40))

leakage_audit = pd.DataFrame([
    ["Airbnb price", "price_quote_*; estimated_revenue_l365d", "Direct target proxy/post-outcome revenue"],
    ["Airbnb superhost", "host_is_superhost raw label", "Direct duplicate of classification target"],
    ["NYC resolution", "Closed Date; Status; Resolution Description", "Only known after or during resolution"],
    ["NYC complaint", "raw complaint field", "Direct source of engineered complaint_category target"],
    ["HR attrition", "EmployeeNumber", "Identifier with no transferable semantics"],
    ["Retail future spending", "future_gross_spend; future_orders; future_return_value", "Computed inside prediction horizon"],
    ["Retail segment", "future_net_spending", "Direct source of segment label"],
], columns=["task", "excluded_fields", "reason"])
display(leakage_audit)

near_constant_table.to_csv(ARTIFACTS / "near_constant_features.csv", index=False)
redundancy_table.to_csv(ARTIFACTS / "redundant_features.csv", index=False)
leakage_audit.to_csv(ARTIFACTS / "leakage_audit.csv", index=False)


In [ ]:
label_quality = pd.DataFrame([
    ["Airbnb price", int((air["price"] >= 10_000).sum()),
     "Advertised price may encode minimum-stay quirks, blocked calendars, scrape errors, or genuine luxury inventory."],
    ["Airbnb superhost", int(air["host_is_superhost"].isna().sum()),
     "Platform-defined status changes over time and missing labels are not valid negatives."],
    ["NYC complaint", int((nyc["complaint_category"] == "OTHER").sum()),
     "Caller-selected and agency-normalized categories overlap semantically; taxonomy changes create temporal label drift."],
    ["NYC resolution", int(nyc["closed_at"].isna().sum()),
     "Open requests are censored, reopened cases and administrative closure can make Closed Date an imperfect service outcome."],
    ["IBM HR attrition", int(y_hr.eq(1).sum()),
     "Small, unusually complete and widely described as synthetic; associations may not transport to a real employer."],
    ["Retail future segment", int((customer["customer_segment"] == "Future_Inactive").sum()),
     "Segments are constructed from a finite future window; inactivity can mean churn, seasonality, or observation-window truncation."],
], columns=["target", "affected_or_example_count", "quality_risk"])
display(label_quality)
label_quality.to_csv(ARTIFACTS / "label_quality_audit.csv", index=False)


# From-scratch model development

All core estimators are implemented in [`src/scratch_ml.py`](src/scratch_ml.py) using NumPy and SciPy only.

For classification, multinomial logistic regression minimizes cross-entropy with an \(L_2\) penalty. For regression, the linear estimator solves

\[
\hat\beta=(X^\top X+\lambda I)^{-1}X^\top y.
\]

KNN uses direct Euclidean distances. The CART decision tree greedily maximizes Gini reduction for classification or variance reduction for regression.

The kernel SVM minimizes regularized squared hinge loss in the RKHS:

\[
\min_{f\in\mathcal H}\ \frac{\lambda}{2}\lVert f\rVert_{\mathcal H}^{2}
+\frac1n\sum_i \max(0,1-y_i f(x_i))^2,
\qquad f(x)=\sum_j\alpha_j K(x,x_j).
\]

Kernel ridge regression uses \(\alpha=(K+\lambda I)^{-1}y\). Kernel KNN computes
\[
d_\phi^2(x,z)=K(x,x)+K(z,z)-2K(x,z).
\]
KPCA eigendecomposes the centered Gram matrix and sends the nonlinear components to the from-scratch logistic classifier.

Regression models are trained on `log1p` targets because price, duration, and spending have extreme right tails. Predictions are transformed back for metrics. A training-only 0.1–99.9% transformed-target guard prevents unstable exponential extrapolation; the clipped fraction is reported.


In [ ]:
from src.homework_experiments import (
    load_prepared_tasks, run_all_experiments, kernel_complexity_table,
    grouped_error_table,
)

tasks = load_prepared_tasks(OUT)
task_inventory = pd.DataFrame([
    {
        "task": name,
        "dataset": task.dataset,
        "type": task.task_type,
        "engineered_rows": task.full_rows,
        "experiment_train": len(task.X_train),
        "experiment_test": len(task.X_test),
        "encoded_features": task.X_train.shape[1],
    }
    for name, task in tasks.items()
])
display(task_inventory)
task_inventory.to_csv(ARTIFACTS / "experiment_task_inventory.csv", index=False)


In [ ]:
algorithm_applicability = pd.DataFrame([
    ["Logistic Regression", "Classification", "Airbnb superhost; NYC complaint; HR attrition; retail segment"],
    ["Linear Regression", "Regression", "Airbnb price; NYC resolution; retail future spending"],
    ["KNN", "Both", "All seven tasks using classifier/regressor variants"],
    ["Decision Tree", "Both", "All seven tasks using Gini/variance variants"],
    ["Kernel SVM", "Classification", "All four datasets through their classification target"],
    ["Kernel Ridge Regression", "Regression", "Airbnb, NYC 311, and retail; HR has no regression target"],
    ["Kernel KNN", "Both", "All seven tasks using classifier/regressor variants"],
    ["KPCA + Logistic", "Classification", "All four datasets through their classification target"],
], columns=["algorithm", "mathematical_scope", "executed_on"])
display(algorithm_applicability)
algorithm_applicability.to_csv(ARTIFACTS / "algorithm_applicability.csv", index=False)


## Controlled experiment design

Every model for a given task receives the same training and test rows. NYC uses chronological holdout; the other tasks use seeded stratified/random holdouts, while HR uses its untouched predefined test set. Exact kernel experiments use controlled subsets because their Gram matrices require \(O(n^2)\) memory and KRR/KPCA require \(O(n^3)\) algebra. The subset is not hidden: its size is reported beside every metric.

The goal is a fair algorithmic comparison at a scale where every required method can actually run from scratch. The computational section separately quantifies why exact kernels cannot consume the complete 311 table.


In [ ]:
results, kernel_geometry_table, predictions, failures = run_all_experiments(
    tasks, output_dir=ARTIFACTS, continue_on_error=False
)
display(results)
assert results.get("error", pd.Series(dtype=object)).isna().all()


In [ ]:
classification_rank = (
    results.loc[results["task_type"].eq("classification")]
    .sort_values(["task", "macro_f1"], ascending=[True, False])
    .groupby("task").head(5)
)
regression_rank = (
    results.loc[results["task_type"].eq("regression")]
    .sort_values(["task", "mae"], ascending=[True, True])
    .groupby("task").head(5)
)
display(classification_rank[["task", "model", "accuracy", "balanced_accuracy", "macro_f1"]])
display(regression_rank[["task", "model", "mae", "rmse", "r2", "prediction_clipped_fraction"]])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=classification_rank, x="task", y="macro_f1", hue="model", ax=axes[0])
axes[0].tick_params(axis="x", rotation=25)
axes[0].set_title("Classification: macro-F1")
sns.barplot(data=regression_rank, x="task", y="mae", hue="model", ax=axes[1])
axes[1].tick_params(axis="x", rotation=25)
axes[1].set_title("Regression: MAE (lower is better)")
plt.tight_layout()
plt.savefig(ARTIFACTS / "model_performance_overview.png", dpi=170)
plt.show()


## Attrition imbalance intervention

Resampling is evaluated rather than assumed beneficial. All variants use the same untouched test set and the same preprocessing map fitted on the original training data. Undersampling changes the effective prior and discards information; mixed SMOTE preserves the majority sample but creates interpolated minority employees. Macro-F1 and balanced accuracy reveal tradeoffs hidden by ordinary accuracy.


In [ ]:
hr_variants = {
    "original": pd.read_csv(OUT / "hr_train_original.csv"),
    "undersampled": pd.read_csv(OUT / "hr_train_undersampled.csv"),
    "mixed_smote": pd.read_csv(OUT / "hr_train_mixed_smote.csv"),
}
hr_test_final = pd.read_csv(OUT / "hr_test_untouched.csv")
hr_feature_cols = [c for c in hr_test_final if c != "Attrition"]
hr_prep = TabularPreprocessor(max_categories=20, min_category_count=3).fit(
    hr_variants["original"][hr_feature_cols]
)
X_hr_eval = hr_prep.transform(hr_test_final[hr_feature_cols])
y_hr_eval = hr_test_final["Attrition"].to_numpy(int)
imbalance_rows = []
for variant_name, variant in hr_variants.items():
    Xv = hr_prep.transform(variant[hr_feature_cols])
    yv = variant["Attrition"].to_numpy(int)
    for model_name, model in {
        "Logistic Regression": LogisticRegressionScratch(
            learning_rate=.03, epochs=300, l2=1e-2
        ),
        "Decision Tree": DecisionTreeScratch(
            max_depth=7, min_samples_split=20, min_samples_leaf=6,
            max_features=min(40, Xv.shape[1]), random_state=SEED
        ),
    }.items():
        pred = model.fit(Xv, yv).predict(X_hr_eval)
        imbalance_rows.append({
            "training_variant": variant_name, "model": model_name,
            "train_rows": len(variant), **classification_metrics(y_hr_eval, pred)
        })
imbalance_comparison = pd.DataFrame(imbalance_rows)
display(imbalance_comparison.sort_values("macro_f1", ascending=False))
imbalance_comparison.to_csv(ARTIFACTS / "hr_resampling_comparison.csv", index=False)


# Kernel investigation: geometry, not score narration

Three quantities connect performance to the manifold:

1. **Centered kernel alignment** measures agreement between the Gram matrix and target similarity.
2. **Effective rank** measures how many independent directions the kernel preserves.
3. **Off-diagonal concentration** reveals whether examples become almost identical or almost orthogonal.

These are interpreted with the model metrics. A high alignment with extremely low effective rank may still underfit; a very high effective rank on a small sample may overfit. Kernel choice is therefore dataset-specific.


In [ ]:
display(kernel_geometry_table.sort_values(["task", "alignment"], ascending=[True, False]))

kernel_results = results.loc[
    results["model"].str.contains("Kernel|KRR|KPCA", regex=True)
].copy()
best_kernel_by_task = []
for task_name, group in kernel_results.groupby("task"):
    if group["task_type"].iloc[0] == "classification":
        best = group.sort_values(["macro_f1", "balanced_accuracy"], ascending=False).iloc[0]
    else:
        best = group.sort_values("mae").iloc[0]
    best_kernel_by_task.append(best)
best_kernel_by_task = pd.DataFrame(best_kernel_by_task)
display(best_kernel_by_task[[
    "task", "model", "macro_f1", "balanced_accuracy", "mae", "rmse",
    "fit_seconds", "peak_python_memory_mb"
]])

kernel_results["kernel"] = kernel_results["model"].str.extract(r"\[(linear|poly|rbf)\]")[0]
kernel_analysis = kernel_results.merge(
    kernel_geometry_table, on=["task", "kernel"], how="left"
)
kernel_analysis.to_csv(ARTIFACTS / "kernel_performance_and_geometry.csv", index=False)


### Dataset-specific interpretation guide

- **Airbnb:** geographic coordinates, room type, city, amenities, capacity, and reviews interact. Polynomial/RBF gains indicate that price or superhost behavior is not additive. A weak RBF result can also mean the median bandwidth made mixed one-hot and continuous points too uniformly similar.
- **NYC 311:** agency, complaint family, borough, channel, hour, and request age create localized operational regimes. A nonlinear kernel is useful only if it preserves rare-class neighborhoods; high accuracy with low macro-F1 means majority complaint geometry dominated.
- **IBM HR:** the sample is small and mostly low-cardinality. If the linear kernel wins, the kernel trick added variance without enough observations to estimate a curved boundary. A better balanced accuracy with lower ordinary accuracy can be desirable under attrition imbalance.
- **Online Retail:** RFM behavior, returns, and trend ratios create nonlinear thresholds. Polynomial interactions can capture “high frequency but declining spend” differently from “low frequency but growing spend.” Exact RBF can still underperform when inactive customers form a point mass and active spending has a very long tail.
- **Kernel KNN caveat:** with a stationary RBF kernel, \(d_\phi^2=2-2e^{-\gamma\lVert x-z\rVert^2}\), a monotone transformation of Euclidean distance. Neighbor ordering is therefore unchanged; only weighting changes. Large gains should not be expected unless the kernel changes geometry rather than merely rescaling distance.


In [ ]:
nonkernel_names = {
    "Logistic Regression", "Linear Regression", "KNN", "Decision Tree"
}
kernel_justification_rows = []
rationale = {
    "airbnb_price": "Price depends on interactions among geography, room type, capacity, and amenities; additive effects are insufficient in heterogeneous city markets.",
    "airbnb_superhost": "Host status combines review history, responsiveness, tenure, and listing structure; local neighborhoods can matter but label rules constrain achievable gains.",
    "nyc_complaint": "Agency, borough, channel, and cyclic time variables define operational regimes; rare classes require geometry that does not collapse into dominant complaints.",
    "nyc_resolution": "Resolution processes vary nonlinearly by agency, request timing, channel, and geography, producing localized duration surfaces.",
    "hr_attrition": "The sample is small and low-cardinality; additional RKHS flexibility can increase variance, while imbalance favors a comparatively stable linear boundary.",
    "retail_segment": "RFM, returns, and spending trends interact, but the constructed tiers overlap and inactivity is a point mass rather than a smooth manifold.",
    "retail_spending": "Zero inflation and extreme active-customer tails violate a single smooth regression surface; a tree can isolate thresholds more directly than a global exact kernel.",
}
for task_name, group in results.groupby("task"):
    kernel_group = group.loc[~group["model"].isin(nonkernel_names)]
    nonkernel_group = group.loc[group["model"].isin(nonkernel_names)]
    if group["task_type"].iloc[0] == "classification":
        best_kernel = kernel_group.sort_values("macro_f1", ascending=False).iloc[0]
        best_non = nonkernel_group.sort_values("macro_f1", ascending=False).iloc[0]
        benefit = best_kernel["macro_f1"] - best_non["macro_f1"]
        metric = "macro_f1"
    else:
        best_kernel = kernel_group.sort_values("mae").iloc[0]
        best_non = nonkernel_group.sort_values("mae").iloc[0]
        benefit = (best_non["mae"] - best_kernel["mae"]) / max(best_non["mae"], 1e-12)
        metric = "relative_MAE_reduction"
    geom = kernel_geometry_table.loc[
        kernel_geometry_table["task"].eq(task_name)
    ].sort_values("alignment", ascending=False).iloc[0]
    kernel_justification_rows.append({
        "task": task_name,
        "best_kernel_model": best_kernel["model"],
        "best_non_kernel_model": best_non["model"],
        "comparison_metric": metric,
        "kernel_benefit": benefit,
        "highest_alignment_kernel": geom["kernel"],
        "highest_alignment": geom["alignment"],
        "effective_rank_at_highest_alignment": geom["effective_rank"],
        "dataset_specific_reason": rationale[task_name],
    })
kernel_justification_table = pd.DataFrame(kernel_justification_rows)
display(kernel_justification_table.sort_values("kernel_benefit", ascending=False))
kernel_justification_table.to_csv(ARTIFACTS / "kernel_justification_table.csv", index=False)


# Cross-dataset synthesis

Cleaning burden and feature-engineering burden are different. NYC 311 is the hardest to clean because scale, temporal drift, repeated reports, evolving complaint taxonomy, geospatial missingness, and right-censored resolution coexist. Online Retail requires the most aggressive feature engineering because the raw observational unit is a transaction line while both targets live at the future customer level.


In [ ]:
cross_dataset_evidence = pd.DataFrame([
    ["Airbnb", len(air_raw), air_raw.isna().mean().max(), "Mixed listing/host/geographic table",
     "Train-only imputation, compact missing indicators, monetary parsing, robust target variants"],
    ["NYC 311", total_rows, nyc.isna().mean().max(), "Large drifting event log",
     "Chunking, hash sampling, duplicate clusters, cyclic time, censoring"],
    ["IBM HR", len(hr_raw), hr_raw.isna().mean().max(), "Small clean imbalanced table",
     "Constant removal, domain ratios, training-only selection and resampling"],
    ["Online Retail", len(retail_raw), retail_raw.isna().mean().max(), "Transaction lines",
     "Returns/cancellations, chronological cutoff, customer aggregation, future labels"],
], columns=[
    "dataset", "raw_rows", "maximum_missing_rate", "raw_topology",
    "principal_engineering_work"
])
display(cross_dataset_evidence)

dataset_kernel_benefit = (
    kernel_justification_table.assign(
        dataset=kernel_justification_table["task"].map({
            "airbnb_price": "Airbnb", "airbnb_superhost": "Airbnb",
            "nyc_complaint": "NYC 311", "nyc_resolution": "NYC 311",
            "hr_attrition": "IBM HR",
            "retail_segment": "Online Retail", "retail_spending": "Online Retail",
        })
    ).groupby("dataset")["kernel_benefit"].mean().sort_values(ascending=False)
)
display(dataset_kernel_benefit.to_frame("mean_normalized_kernel_benefit"))
cross_dataset_evidence.to_csv(ARTIFACTS / "cross_dataset_engineering_evidence.csv", index=False)
dataset_kernel_benefit.to_csv(ARTIFACTS / "cross_dataset_kernel_benefit.csv")


# Failure analysis: ten worst predictions per task

The selected model is the best on macro-F1 for classification or MAE for regression. Classification errors are prioritized from the misclassified set; regression errors are ranked by absolute error. The tables preserve original engineered features so each case can be traced to a rare class, tail target, sparse history, geographic regime, or ambiguous label.


In [ ]:
failure_summary = []
for task_name, table in failures.items():
    print("\n", task_name)
    display(table)
    task = tasks[task_name]
    if task.task_type == "classification":
        train_counts = pd.Series(task.y_train_model).value_counts()
        table = table.copy()
        table["diagnostic"] = table["actual"].map(
            lambda c: "rare/underrepresented class" if train_counts.get(c, 0) < 30
            else "class overlap, drift, or ambiguous label"
        )
    else:
        table = table.copy()
        q99 = np.quantile(task.y_test_eval, .99)
        table["diagnostic"] = np.where(
            table["actual"] >= q99,
            "extreme right-tail target / extrapolation limit",
            "feature collision or unobserved explanatory variable",
        )
    table.insert(0, "task", task_name)
    failure_summary.append(table)
failure_summary_table = pd.concat(failure_summary, ignore_index=True, sort=False)
failure_summary_table.to_csv(ARTIFACTS / "failure_postmortem_all_tasks.csv", index=False)


Systemic failures repeat across many cases with similar geometry—for example rare 311 complaints being mapped to dominant classes or zero-inflated retail behavior. Data artifacts are isolated cases such as implausible advertised prices, administrative closures, or extreme returns. The final paper should cite both the row-level evidence above and aggregate class/group behavior before assigning either explanation.


# Bias, drift, and real-world reliability

Fairness is broader than protected-class parity here. Airbnb and 311 have geographic representation bias; 311 also has channel access and temporal drift. HR exposes gender but omits many protected and organizational variables, making a clean fairness conclusion impossible. Retail is dominated by the United Kingdom and by identified repeat customers; anonymous customers were necessarily excluded.


In [ ]:
best_predictions = {}
bias_tables = []
group_map = {
    "airbnb_price": ["city", "room_type"],
    "airbnb_superhost": ["city", "room_type"],
    "nyc_complaint": ["Borough", "Open Data Channel Type", "created_month"],
    "nyc_resolution": ["Borough", "Open Data Channel Type", "created_month"],
    "hr_attrition": ["Gender", "Department", "OverTime"],
    "retail_segment": ["country"],
    "retail_spending": ["country"],
}
for task_name, task in tasks.items():
    subset = results.loc[results["task"].eq(task_name)]
    if task.task_type == "classification":
        best_model = subset.sort_values(["macro_f1", "balanced_accuracy"], ascending=False).iloc[0]["model"]
    else:
        best_model = subset.sort_values("mae").iloc[0]["model"]
    pred = predictions[(task_name, best_model)]
    best_predictions[task_name] = pred
    for group_col in group_map[task_name]:
        table = grouped_error_table(task, pred, group_col)
        if len(table):
            table.insert(0, "task", task_name)
            table.insert(1, "selected_model", best_model)
            bias_tables.append(table)
bias_group_metrics = pd.concat(bias_tables, ignore_index=True, sort=False)
display(bias_group_metrics)
bias_group_metrics.to_csv(ARTIFACTS / "bias_group_metrics.csv", index=False)

distribution_bias = pd.DataFrame([
    ["Airbnb", "Geographic", air_raw["city"].value_counts(normalize=True).max(),
     "The largest city dominates; conclusions may reflect its inventory and regulations."],
    ["NYC 311", "Complaint/channel/time", nyc["Problem (formerly Complaint Type)"].value_counts(normalize=True).max(),
     "Complaint propensity measures service access and reporting behavior, not only underlying incidents."],
    ["IBM HR", "Class and omitted-group", y_hr.eq(0).mean(),
     "Non-attrition dominates and protected attributes are incomplete."],
    ["Online Retail", "Country/identity", retail["Country"].eq("United Kingdom").mean(),
     "UK and identified customers dominate; guest transactions are excluded from customer models."],
], columns=["dataset", "bias_axis", "dominant_fraction", "reliability_consequence"])
display(distribution_bias)
distribution_bias.to_csv(ARTIFACTS / "distribution_bias_summary.csv", index=False)


# Computational analysis

Timing is empirical for the controlled experiment subsets. `tracemalloc` measures Python-managed peak memory and therefore undercounts native BLAS/NumPy buffers; the analytical Gram-matrix calculation supplies the missing order-of-magnitude argument.


In [ ]:
timing_summary = results[[
    "task", "model", "train_rows", "feature_dimension",
    "fit_seconds", "predict_seconds", "peak_python_memory_mb"
]].sort_values("fit_seconds", ascending=False)
display(timing_summary.head(25))

complexity = kernel_complexity_table(tasks)
display(complexity)
complexity.to_csv(ARTIFACTS / "kernel_complexity.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 6))
plot_timing = timing_summary.copy()
plot_timing["fit_seconds_plot"] = plot_timing["fit_seconds"].clip(lower=1e-4)
sns.scatterplot(
    data=plot_timing, x="train_rows", y="fit_seconds_plot",
    hue="model", style="task", s=80, ax=ax
)
ax.set_yscale("log")
ax.set_title("Training time grows sharply for exact kernel algebra")
plt.tight_layout()
plt.savefig(ARTIFACTS / "training_time_scaling.png", dpi=170)
plt.show()


For \(n\) observations, storing one dense float64 Gram matrix costs \(8n^2\) bytes. KRR and KPCA dense decompositions are \(O(n^3)\). The complete NYC engineered sample therefore cannot be treated like a textbook dataset: even before cross-validation or multiple kernels, the matrix can exceed ordinary workstation memory and the factorization becomes prohibitive. Subsampling is part of the computational experiment, not an attempt to conceal poor scaling.

# Theory versus reality

Iris and standard MNIST benchmarks usually provide a fixed, mostly complete matrix; a stable taxonomy; little censoring; negligible identity resolution; and train/test sets drawn from closely matched distributions. These four datasets violate those assumptions:

- Airbnb has source-dependent missingness, geographic regimes, platform-defined labels, and genuine extreme prices.
- 311 has evolving categories, repeated reports, channel bias, temporal drift, and unresolved right-censored outcomes.
- HR is small and imbalanced, so a few minority errors dominate operational usefulness even when accuracy looks high.
- Retail begins as transaction lines, contains returns and anonymous purchases, and requires a temporal definition of both the customer and the future target.

The experiments provide direct evidence: macro-F1 diverges from accuracy under imbalance; error concentrates in rare classes and target tails; preprocessing choices alter geometry; and exact kernel complexity forces a smaller experimental regime. Textbook performance is optimistic because the benchmark has already solved much of the data-definition problem.

# Reproducible outputs for the paper

All report-ready tables and figures are in `artifacts/`; engineered task tables are in `processed/model_tables/`. The final LaTeX paper should consume these outputs rather than manually transcribing numbers. The two fully commented object-oriented source files—`src/scratch_ml.py` and `src/homework_experiments.py`—form the code appendix together with this notebook.


In [ ]:
artifact_inventory = pd.DataFrame([
    {"file": p.name, "size_kb": round(p.stat().st_size / 1024, 1)}
    for p in sorted(ARTIFACTS.glob("*")) if p.is_file()
])
display(artifact_inventory)
assert (ARTIFACTS / "model_results.csv").exists()
assert (ARTIFACTS / "kernel_geometry.csv").exists()
assert (ARTIFACTS / "failure_postmortem_all_tasks.csv").exists()
assert (ARTIFACTS / "bias_group_metrics.csv").exists()
print("Integrated homework notebook completed successfully.")
